In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, numpy as np, pandas as pd, pyfixest as pf, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from scipy.stats import norm
import pyarrow.parquet as pq
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 240); pd.set_option('display.max_columns', 60)
plt.rcParams.update({'font.sans-serif': ['Arial', 'DejaVu Sans'],
                     'axes.unicode_minus': False})


In [ ]:
GRID   = 2                                    # int, do not change to 2.0 (CACHE path is built from it)
CACHE  = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid{GRID}'
ACC_FP = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
panel  = pd.read_parquet(rf'{CACHE}\panel_global.parquet')

# ══════════════ model specification ══════════════
Y_MAIN, Y_ALT = 'A_it', 'A_wx'                # ← main definition = all accidents; A_wx demoted to robustness
OFFSET, CLUSTER = 'log_E', 'grid_id'

FE_ABS    = 'grid_id^month + region^year_month'   # absolute module: treatment is seasonal, needs grid^month
FE_REL    = 'grid_id + region^year_month'         # relative module: treatment already season-normalised
FE_REL_WI = ['grid_id + year_month',              # within-region version (region^ym degenerates)
             'grid_id^month + year_month', 'grid_id']

TYPES_ALL = ['Machinery','Grounding','Collision','Contact','Fire_Explosion',
             'Foundered','Hull_Damage','Piracy_Other','Misc']
TYPES_WX  = ['Grounding','Collision','Contact','Foundered','Hull_Damage']
NEG_CTRL = ['Piracy_Other']          # negative control
DROP_CORE = ['Misc','Piracy_Other']               # removed from A_it to obtain A_core

panel['A_core'] = panel['A_it'] - panel[[f'A_{t}' for t in DROP_CORE]].sum(axis=1)
assert (panel['A_core'] >= 0).all(), 'A_core has negative values'

WV_COLS = [c for c in panel.columns if c.startswith('Wv_')]
WD_COLS = [c for c in panel.columns if c.startswith('Wd_')]
WV_REG, WD_REG = WV_COLS[1:], WD_COLS[1:]         # drop lowest bin as reference category
TAGS = ('P95','P99')
MIN_EVENTS, MIN_CLUSTERS = 60, 50
def rel3(tag): return [f'wind_only_{tag}', f'wave_only_{tag}', f'compound_{tag}']

# ══════════════ utilities ══════════════
def stars(pv): return '***' if pv<.001 else '**' if pv<.01 else '*' if pv<.05 else '†' if pv<.1 else ''
def active(p, col='A_it', min_acc=1):
    return p[p.groupby('grid_id')[col].transform('sum') >= min_acc].copy()
def _vc(m):
    for at in ('_vcov','vcov_'):
        if hasattr(m, at): return np.asarray(getattr(m, at))
def contrast(m, pos, neg=()):
    n = list(m.coef().index); b = m.coef().values; V = _vc(m)
    c = np.zeros(len(n))
    for x in pos: c[n.index(x)] += 1
    for x in neg: c[n.index(x)] -= 1
    e = float(c @ b); s = float(np.sqrt(c @ V @ c))
    return e, s, 2*(1 - norm.cdf(abs(e/s)))
def fitm(y, xs, data, fe, cluster=CLUSTER, offset=OFFSET):
    kw = dict(data=data, vcov={'CRV1': cluster})
    if offset: kw['offset'] = offset
    return pf.fepois(f"{y} ~ {' + '.join(xs)} | {fe}", **kw)
def fit_ladder(y, xs, data, ladder, cluster=CLUSTER, offset=OFFSET):
    """Step down the FE ladder, return (model, fe); success only if all xs are retained"""
    for fe in ladder:
        try:
            m = fitm(y, xs, data, fe, cluster, offset)
            if all(x in set(m.coef().index) for x in xs): return m, fe
        except Exception: continue
    return None, None
def row(m, x):
    b, se = float(m.coef()[x]), float(m.se()[x]); pv = float(m.pvalue()[x])
    return np.exp(b), np.exp(b-1.96*se), np.exp(b+1.96*se), pv

RES = []
def log(block, label, irr, lo, hi, p, n=None, note=''):
    RES.append(dict(block=block, label=label, IRR=irr, lo=lo, hi=hi, p=p, n=n, note=note))
T1 = T2 = T3 = T4 = SCAN = pd.DataFrame()

PA = active(panel)
print('══════ model specification ══════')
print(f'  dependent var  main {Y_MAIN} ({int(PA[Y_MAIN].sum()):,})   robust {Y_ALT} ({int(PA[Y_ALT].sum()):,})'
      f'   core A_core ({int(PA["A_core"].sum()):,})')
print(f'  offset {OFFSET} | cluster {CLUSTER}')
print(f'  absolute module FE  {FE_ABS}')
print(f'  relative module FE  {FE_REL} (appendix only)')
print(f'  sample {len(PA):,} rows / {PA["grid_id"].nunique()} cells / {panel["year_month"].nunique()} months')


In [ ]:
print('='*100); print('S1  panel data checks')

print('\n[Structure]')
bal = panel.groupby('grid_id').size()
print(f'  rows {len(panel):,} | cells {panel["grid_id"].nunique():,} | months {panel["year_month"].nunique()}'
      f' | {panel["year_month"].min()} → {panel["year_month"].max()}')
print(f'  balance: months per cell min={bal.min()} max={bal.max()} complete(132 mo)={int((bal==132).sum())}/{len(bal)}')
print(f'  cell-months with E_it≤0 {int((panel["E_it"]<=0).sum()):,}'
      f' | cell-months with n_days<28 {int((panel["n_days"]<28).sum()):,} (kept in panel; months with incomplete weather days)')
print(f'  wave bins close {(panel[WV_COLS].sum(axis=1).values == panel["n_days"].values).all()}'
      f' | wind bins close {(panel[WD_COLS].sum(axis=1).values == panel["n_days"].values).all()}')

print('\n[Outcome variables]')
tsum = int(panel[[f'A_{t}' for t in TYPES_ALL]].sum().sum())
print(f'  A_it = {int(panel["A_it"].sum()):,}   Σ(9 types) = {tsum:,}   MECE {"✓" if tsum==panel["A_it"].sum() else "✗"}')
print(f'  A_wx = {int(panel["A_wx"].sum()):,}   A_core (excl. {"+".join(DROP_CORE)}) = {int(panel["A_core"].sum()):,}')
for t in TYPES_ALL:
    n = int(panel[f'A_{t}'].sum())
    flag = '★weather' if t in TYPES_WX else '◆neg. control' if t in NEG_CTRL else '○'
    print(f'    A_{t:16s} {n:6,d}  ({n/panel["A_it"].sum()*100:5.2f}%)  {flag}')

print('\n[Support by region]')
g = panel.groupby(['region','subregion']).agg(
        cells=('grid_id','nunique'), rows=('A_it','size'),
        A_it=('A_it','sum'), A_wx=('A_wx','sum'), E=('E_it','sum'))
g['accidents_per_cell_month'] = (g['A_it']/g['rows']).round(4)
print(g.to_string())
print(f'\n  ⚠ regions with cells<{MIN_CLUSTERS} (clustered SE unreliable):',
      [r for r, n in panel.groupby('region')['grid_id'].nunique().items() if n < MIN_CLUSTERS] or 'none')


In [ ]:
GRID   = 2                                    # int, do not change to 2.0 (CACHE path is built from it)
CACHE  = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid{GRID}'
ACC_FP = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
panel  = pd.read_parquet(rf'{CACHE}\panel_global.parquet')

# ══════════════ model specification ══════════════
Y_MAIN, Y_ALT = 'A_it', 'A_wx'                # ← main definition = all accidents; A_wx demoted to robustness
OFFSET, CLUSTER = 'log_E', 'grid_id'

FE_ABS    = 'grid_id^month + region^year_month'   # absolute module: treatment is seasonal, needs grid^month
FE_REL    = 'grid_id + region^year_month'         # relative module: treatment already season-normalised
FE_REL_WI = ['grid_id + year_month',              # within-region version (region^ym degenerates)
             'grid_id^month + year_month', 'grid_id']

TYPES_ALL = ['Machinery','Grounding','Collision','Contact','Fire_Explosion',
             'Foundered','Hull_Damage','Piracy_Other','Misc']
TYPES_WX  = ['Grounding','Collision','Contact','Foundered','Hull_Damage']
NEG_CTRL  = ['Machinery','Piracy_Other']          # negative control
DROP_CORE = ['Misc','Piracy_Other']               # removed from A_it to obtain A_core

panel['A_core'] = panel['A_it'] - panel[[f'A_{t}' for t in DROP_CORE]].sum(axis=1)
assert (panel['A_core'] >= 0).all(), 'A_core has negative values'

WV_COLS = [c for c in panel.columns if c.startswith('Wv_')]
WD_COLS = [c for c in panel.columns if c.startswith('Wd_')]
WV_REG, WD_REG = WV_COLS[1:], WD_COLS[1:]         # drop lowest bin as reference category
TAGS = ('P95','P99')
MIN_EVENTS, MIN_CLUSTERS = 60, 50
def rel3(tag): return [f'wind_only_{tag}', f'wave_only_{tag}', f'compound_{tag}']

# ══════════════ utilities ══════════════
def stars(pv): return '***' if pv<.001 else '**' if pv<.01 else '*' if pv<.05 else '†' if pv<.1 else ''
def active(p, col='A_it', min_acc=1):
    return p[p.groupby('grid_id')[col].transform('sum') >= min_acc].copy()
def _vc(m):
    for at in ('_vcov','vcov_'):
        if hasattr(m, at): return np.asarray(getattr(m, at))
def contrast(m, pos, neg=()):
    n = list(m.coef().index); b = m.coef().values; V = _vc(m)
    c = np.zeros(len(n))
    for x in pos: c[n.index(x)] += 1
    for x in neg: c[n.index(x)] -= 1
    e = float(c @ b); s = float(np.sqrt(c @ V @ c))
    return e, s, 2*(1 - norm.cdf(abs(e/s)))
def fitm(y, xs, data, fe, cluster=CLUSTER, offset=OFFSET):
    kw = dict(data=data, vcov={'CRV1': cluster})
    if offset: kw['offset'] = offset
    return pf.fepois(f"{y} ~ {' + '.join(xs)} | {fe}", **kw)
def fit_ladder(y, xs, data, ladder, cluster=CLUSTER, offset=OFFSET):
    """Step down the FE ladder, return (model, fe); success only if all xs are retained"""
    for fe in ladder:
        try:
            m = fitm(y, xs, data, fe, cluster, offset)
            if all(x in set(m.coef().index) for x in xs): return m, fe
        except Exception: continue
    return None, None
def row(m, x):
    b, se = float(m.coef()[x]), float(m.se()[x]); pv = float(m.pvalue()[x])
    return np.exp(b), np.exp(b-1.96*se), np.exp(b+1.96*se), pv

RES = []
def log(block, label, irr, lo, hi, p, n=None, note=''):
    RES.append(dict(block=block, label=label, IRR=irr, lo=lo, hi=hi, p=p, n=n, note=note))
T1 = T2 = T3 = T4 = SCAN = pd.DataFrame()

PA = active(panel)
print('══════ model specification ══════')
print(f'  dependent var  main {Y_MAIN} ({int(PA[Y_MAIN].sum()):,})   robust {Y_ALT} ({int(PA[Y_ALT].sum()):,})'
      f'   core A_core ({int(PA["A_core"].sum()):,})')
print(f'  offset {OFFSET} | cluster {CLUSTER}')
print(f'  absolute module FE  {FE_ABS}')
print(f'  relative module FE  {FE_REL} (appendix only)')
print(f'  sample {len(PA):,} rows / {PA["grid_id"].nunique()} cells / {panel["year_month"].nunique()} months')


In [ ]:
print('='*100); print('S1  panel data checks')

print('\n[Structure]')
bal = panel.groupby('grid_id').size()
print(f'  rows {len(panel):,} | cells {panel["grid_id"].nunique():,} | months {panel["year_month"].nunique()}'
      f' | {panel["year_month"].min()} → {panel["year_month"].max()}')
print(f'  balance: months per cell min={bal.min()} max={bal.max()} complete(132 mo)={int((bal==132).sum())}/{len(bal)}')
print(f'  cell-months with E_it≤0 {int((panel["E_it"]<=0).sum()):,}'
      f' | cell-months with n_days<28 {int((panel["n_days"]<28).sum()):,} (kept in panel; months with incomplete weather days)')
print(f'  wave bins close {(panel[WV_COLS].sum(axis=1).values == panel["n_days"].values).all()}'
      f' | wind bins close {(panel[WD_COLS].sum(axis=1).values == panel["n_days"].values).all()}')

print('\n[Outcome variables]')
tsum = int(panel[[f'A_{t}' for t in TYPES_ALL]].sum().sum())
print(f'  A_it = {int(panel["A_it"].sum()):,}   Σ(9 types) = {tsum:,}   MECE {"✓" if tsum==panel["A_it"].sum() else "✗"}')
print(f'  A_wx = {int(panel["A_wx"].sum()):,}   A_core (excl. {"+".join(DROP_CORE)}) = {int(panel["A_core"].sum()):,}')
for t in TYPES_ALL:
    n = int(panel[f'A_{t}'].sum())
    flag = '★weather' if t in TYPES_WX else '◆neg. control' if t in NEG_CTRL else '○'
    print(f'    A_{t:16s} {n:6,d}  ({n/panel["A_it"].sum()*100:5.2f}%)  {flag}')

print('\n[Support by region]')
g = panel.groupby(['region','subregion']).agg(
        cells=('grid_id','nunique'), rows=('A_it','size'),
        A_it=('A_it','sum'), A_wx=('A_wx','sum'), E=('E_it','sum'))
g['accidents_per_cell_month'] = (g['A_it']/g['rows']).round(4)
print(g.to_string())
print(f'\n  ⚠ regions with cells<{MIN_CLUSTERS} (clustered SE unreliable):',
      [r for r, n in panel.groupby('region')['grid_id'].nunique().items() if n < MIN_CLUSTERS] or 'none')


In [ ]:
print('='*100); print('S2  load dmax + threshold-sweep variables')

def load_dmax():
    """Daily max Hs / wind speed for all regions, read once"""
    fr = []
    for r in panel['region'].unique():
        w = pd.read_parquet(rf'{CACHE}\{r}__dmax_wave.parquet').rename(columns={'dmax':'swh'})
        i = pd.read_parquet(rf'{CACHE}\{r}__dmax_wind.parquet').rename(columns={'dmax':'wsp'})
        fr.append(w.merge(i, on=['grid_id','date'], how='inner'))
    d = pd.concat(fr, ignore_index=True).drop_duplicates(['grid_id','date'])
    d = d[np.isfinite(d['swh']) & np.isfinite(d['wsp'])          # SWH=NaN under sea ice, must be removed
          & d['grid_id'].isin(set(panel['grid_id']))].copy()
    d['date'] = pd.to_datetime(d['date'])
    return d.reset_index(drop=True)

DM = load_dmax()
print(f'  DM {len(DM):,} rows / {DM["grid_id"].nunique()} cells / '
      f'{DM["date"].min().date()} → {DM["date"].max().date()}  '
      f'({DM.memory_usage(deep=True).sum()/1e6:.0f} MB)')

WAVE_CUTS = [1.25, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
WIND_CUTS = [8.0, 10.0, 12.0, 13.9, 15.0, 17.2, 19.0, 20.8, 22.0]
def cutname(pre, c): return f'{pre}_ge{c:g}'.replace('.', 'p')

SCAN_CP = rf'{CACHE}\_scan_days.parquet'
need = [cutname('wv', c) for c in WAVE_CUTS] + [cutname('wd', c) for c in WIND_CUTS]
if os.path.exists(SCAN_CP) and set(need) <= set(pq.read_schema(SCAN_CP).names):
    S = pd.read_parquet(SCAN_CP); print('  read cache', SCAN_CP)
else:
    d = DM.copy(); d['year_month'] = d['date'].dt.strftime('%Y-%m')
    for c in WAVE_CUTS: d[cutname('wv', c)] = (d['swh'] >= c).astype('int8')
    for c in WIND_CUTS: d[cutname('wd', c)] = (d['wsp'] >= c).astype('int8')
    cols = [c for c in d.columns if c.startswith(('wv_ge','wd_ge'))]
    S = d.groupby(['grid_id','year_month'])[cols].sum().reset_index()
    S.to_parquet(SCAN_CP, index=False); print('  cache written', SCAN_CP)

SC = [c for c in S.columns if c not in ('grid_id','year_month')]
panel = panel.drop(columns=[c for c in SC if c in panel.columns])   # idempotent
panel = panel.merge(S, on=['grid_id','year_month'], how='left')
panel[SC] = panel[SC].fillna(0).astype('int16')
PA = active(panel)
print(f'  {len(SC)} sweep variables | wv_ge6 days {int(panel["wv_ge6"].sum()):,}'
      f' | wd_ge17p2 days {int(panel["wd_ge17p2"].sum()):,}')


In [ ]:
print('='*100); print('S3  absolute-threshold compound-day variables')
ABS_LADDER = [(1.25, 8.0), (2.5, 10.8), (4.0, 13.9), (6.0, 17.2), (9.0, 20.8)]   # panel c: matched wind/wave ladder
ABS_D      = [(4.0, 13.9), (6.0, 17.2)]                                        # panel d: two levels
ABS_GRID   = ABS_LADDER + [(4.0, 17.2), (6.0, 13.9), (6.0, 20.8)]              # appendix sensitivity
LAD_NAME   = {(1.25, 8.0): 'Moderate\nBft 5', (2.5, 10.8): 'Rough\nBft 6', (4.0, 13.9): 'Very rough\nBft 7',
              (6.0, 17.2): 'High\nBft 8', (9.0, 20.8): 'Very high\nBft 9'}
def abstag(hs, u): return f'{hs:g}_{u:g}'.replace('.', 'p')
def abs3(hs, u):  t = abstag(hs, u); return [f'wo_{t}', f'wv_{t}', f'cp_{t}']
def abslab(hs, u): return f'Hs ≥ {hs:g} m & U₁₀ ≥ {u:g} m s⁻¹'

ABS_CP = rf'{CACHE}\_abs_compound_days.parquet'
need = sum((abs3(*c) for c in ABS_GRID), [])
if os.path.exists(ABS_CP) and set(need) <= set(pq.read_schema(ABS_CP).names):
    S = pd.read_parquet(ABS_CP); print('  read cache', ABS_CP)
else:
    d = DM.copy(); d['year_month'] = d['date'].dt.strftime('%Y-%m')
    for hs, u in ABS_GRID:
        ew, ei = d['swh'] >= hs, d['wsp'] >= u
        wo, wv, cp = abs3(hs, u)
        d[wo] = (ei & ~ew).astype('int8'); d[wv] = (ew & ~ei).astype('int8'); d[cp] = (ew & ei).astype('int8')
    S = d.groupby(['grid_id', 'year_month'])[need].sum().reset_index()
    S.to_parquet(ABS_CP, index=False); print('  cache written', ABS_CP)

panel = panel.drop(columns=[c for c in need if c in panel.columns]).merge(S, on=['grid_id','year_month'], how='left')
panel[need] = panel[need].fillna(0).astype('int16')
PA = active(panel)

print('\n  [Panel support for each threshold pair]')
print(f'  {"pair":22s}{"wind_only d":>12s}{"wave_only d":>12s}{"compound d":>12s}{"cp>0 c-m":>10s}{"accidents":>10s}')
for hs, u in ABS_GRID:
    wo, wv, cp = abs3(hs, u)
    print(f'  Hs≥{hs:<4g} U≥{u:<6g}       {int(PA[wo].sum()):>12,}{int(PA[wv].sum()):>12,}{int(PA[cp].sum()):>12,}'
          f'{int((PA[cp]>0).sum()):>10,}{int(PA.loc[PA[cp]>0, Y_MAIN].sum()):>10,}')
for thr in ABS_D:
    cp = abs3(*thr)[2]
    print(f'\n  [{abslab(*thr)} by region]')
    print(PA.groupby('region').agg(cp_days=(cp,'sum'), cp_gm=(cp, lambda s: int((s>0).sum())),
                                   acc=(Y_MAIN,'sum')).sort_values('cp_days', ascending=False).to_string())


In [ ]:
print('='*100); print(f'Table 1  absolute-magnitude dose-response   FE = {FE_ABS}')
rows = []
for y in (Y_MAIN, Y_ALT):
    for cols, unit in [(WV_REG, 'Hs (ref. <1.25m)'), (WD_REG, 'U10 (ref. <8m/s)')]:
        try:
            m = fitm(y, cols, PA, FE_ABS); have = set(m.coef().index)
            for x in cols:
                if x not in have: continue
                irr, lo, hi, pv = row(m, x)
                rows.append(dict(y=y, unit=unit, bin=x, IRR=irr, lo=lo, hi=hi, p=pv))
                log('Table1', f'{y}/{x}', irr, lo, hi, pv, int(PA[y].sum()))
        except Exception as e:
            print(f'  {y}/{unit} FAIL {type(e).__name__}: {e}')
T1 = pd.DataFrame(rows)
for unit in T1['unit'].unique():
    print(f'\n  [{unit}]')
    print(f'    {"bin":16s}{Y_MAIN:>26s}{Y_ALT:>26s}')
    for b in T1[T1['unit'] == unit]['bin'].unique():
        cells = []
        for y in (Y_MAIN, Y_ALT):
            s = T1[(T1['y'] == y) & (T1['bin'] == b)]
            cells.append(f'{s["IRR"].iloc[0]:.4f}[{s["lo"].iloc[0]:.3f},{s["hi"].iloc[0]:.3f}]'
                         f'{stars(s["p"].iloc[0])}' if len(s) else '—')
        print(f'    {b:16s}{cells[0]:>26s}{cells[1]:>26s}')


In [ ]:
print('='*100); print(f'Threshold sweep   FE = {FE_ABS} (separate regression per threshold)')
scan = []
for y in (Y_MAIN, Y_ALT):
    d = active(panel, y)
    for pre, cuts, unit in [('wv', WAVE_CUTS, 'm'), ('wd', WIND_CUTS, 'm/s')]:
        print(f'\n  ### {y} · {"Hs" if pre=="wv" else "U10"}')
        for c in cuts:
            col = cutname(pre, c)
            days  = int(panel[col].sum())
            cells = int((panel.groupby('grid_id')[col].sum() > 0).sum())
            try:
                m = fitm(y, [col], d, FE_ABS)
                irr, lo, hi, pv = row(m, col)
                print(f'    ≥{c:5.1f} {unit:4s} days={days:9,d} cells={cells:4d} '
                      f'IRR={irr:.4f}[{lo:.4f},{hi:.4f}] p={pv:.4f}{stars(pv)}')
                scan.append(dict(y=y, hazard=pre, cut=c, unit=unit, days=days,
                                 cells=cells, IRR=irr, lo=lo, hi=hi, p=pv))
                log('scan', f'{y}/{col}', irr, lo, hi, pv, int(d[y].sum()))
            except Exception as e:
                print(f'    ≥{c:5.1f} {unit:4s} FAIL {type(e).__name__}')
SCAN = pd.DataFrame(scan); assert len(SCAN), 'SCAN is empty'


In [ ]:
print('='*100); print(f'Table 2  single vs compound (absolute thresholds)  FE = {FE_ABS}')
def fit_abs3(y, hs, u, data=PA, fe=FE_ABS):
    xs = abs3(hs, u); wo, wv, cp = xs
    m = fitm(y, xs, data, fe); out = []
    for x, kind in zip(xs, ['wind_only', 'wave_only', 'compound']):
        irr, lo, hi, pv = row(m, x)
        out.append(dict(y=y, hs=hs, u=u, kind=kind, IRR=irr, lo=lo, hi=hi, p=pv))
    e, s, pv = contrast(m, [cp], [wo])                                   # compound vs wind-only
    out.append(dict(y=y, hs=hs, u=u, kind='cp_vs_wo', IRR=np.exp(e), lo=np.exp(e-1.96*s), hi=np.exp(e+1.96*s), p=pv))
    e, s, pv = contrast(m, [cp], [wo, wv])                               # super-additivity, appendix only
    out.append(dict(y=y, hs=hs, u=u, kind='superadd', IRR=np.exp(e), lo=np.exp(e-1.96*s), hi=np.exp(e+1.96*s), p=pv))
    m1 = fitm(y, [cp], data, fe); irr, lo, hi, pv = row(m1, cp)          # univariate: compound vs all other days
    out.append(dict(y=y, hs=hs, u=u, kind='cp_vs_all', IRR=irr, lo=lo, hi=hi, p=pv))
    return out

rows = []
for y in (Y_MAIN, Y_ALT):
    print(f'\n  ### {y}  n={int(PA[y].sum()):,}')
    print(f'    {"pair":18s}{"wind_only":>22s}{"wave_only":>22s}{"compound":>22s}{"cp/wo":>16s}{"cp vs all":>22s}')
    for hs, u in ABS_GRID:
        try: out = fit_abs3(y, hs, u)
        except Exception as e: print(f'    {abslab(hs,u)} FAIL {type(e).__name__}'); continue
        rows += out
        for r in out: log('Table2abs', f'{y}/{abstag(hs,u)}/{r["kind"]}', r['IRR'], r['lo'], r['hi'], r['p'], int(PA[y].sum()))
        g = {r['kind']: r for r in out}
        f = lambda k: f'{g[k]["IRR"]:.3f}[{g[k]["lo"]:.3f},{g[k]["hi"]:.3f}]{stars(g[k]["p"]):3s}'
        print(f'    Hs≥{hs:<4g} U≥{u:<8g}{f("wind_only"):>22s}{f("wave_only"):>22s}{f("compound"):>22s}'
              f'{g["cp_vs_wo"]["IRR"]:>10.3f}{stars(g["cp_vs_wo"]["p"]):<4s}  {f("cp_vs_all"):>22s}')
T2A = pd.DataFrame(rows)
print('\n  compound = three-variable model, reference = days with neither wind nor wave at that level; cp/wo = compound relative to wind-only; cp vs all = univariate')


In [ ]:
print('='*100); print(f'Threshold sweep   FE = {FE_ABS} (separate regression per threshold)')
scan = []
for y in (Y_MAIN, Y_ALT):
    d = active(panel, y)
    for pre, cuts, unit in [('wv', WAVE_CUTS, 'm'), ('wd', WIND_CUTS, 'm/s')]:
        print(f'\n  ### {y} · {"Hs" if pre=="wv" else "U10"}')
        for c in cuts:
            col = cutname(pre, c)
            days  = int(panel[col].sum())
            cells = int((panel.groupby('grid_id')[col].sum() > 0).sum())
            try:
                m = fitm(y, [col], d, FE_ABS)
                irr, lo, hi, pv = row(m, col)
                print(f'    ≥{c:5.1f} {unit:4s} days={days:9,d} cells={cells:4d} '
                      f'IRR={irr:.4f}[{lo:.4f},{hi:.4f}] p={pv:.4f}{stars(pv)}')
                scan.append(dict(y=y, hazard=pre, cut=c, unit=unit, days=days,
                                 cells=cells, IRR=irr, lo=lo, hi=hi, p=pv))
                log('scan', f'{y}/{col}', irr, lo, hi, pv, int(d[y].sum()))
            except Exception as e:
                print(f'    ≥{c:5.1f} {unit:4s} FAIL {type(e).__name__}')
SCAN = pd.DataFrame(scan); assert len(SCAN), 'SCAN is empty'


In [ ]:
print('='*100); print(f'Table 2  single vs compound (absolute thresholds)  FE = {FE_ABS}')
def fit_abs3(y, hs, u, data=PA, fe=FE_ABS):
    xs = abs3(hs, u); wo, wv, cp = xs
    m = fitm(y, xs, data, fe); out = []
    for x, kind in zip(xs, ['wind_only', 'wave_only', 'compound']):
        irr, lo, hi, pv = row(m, x)
        out.append(dict(y=y, hs=hs, u=u, kind=kind, IRR=irr, lo=lo, hi=hi, p=pv))
    e, s, pv = contrast(m, [cp], [wo])                                   # compound vs wind-only
    out.append(dict(y=y, hs=hs, u=u, kind='cp_vs_wo', IRR=np.exp(e), lo=np.exp(e-1.96*s), hi=np.exp(e+1.96*s), p=pv))
    e, s, pv = contrast(m, [cp], [wo, wv])                               # super-additivity, appendix only
    out.append(dict(y=y, hs=hs, u=u, kind='superadd', IRR=np.exp(e), lo=np.exp(e-1.96*s), hi=np.exp(e+1.96*s), p=pv))
    m1 = fitm(y, [cp], data, fe); irr, lo, hi, pv = row(m1, cp)          # univariate: compound vs all other days
    out.append(dict(y=y, hs=hs, u=u, kind='cp_vs_all', IRR=irr, lo=lo, hi=hi, p=pv))
    return out

rows = []
for y in (Y_MAIN, Y_ALT):
    print(f'\n  ### {y}  n={int(PA[y].sum()):,}')
    print(f'    {"pair":18s}{"wind_only":>22s}{"wave_only":>22s}{"compound":>22s}{"cp/wo":>16s}{"cp vs all":>22s}')
    for hs, u in ABS_GRID:
        try: out = fit_abs3(y, hs, u)
        except Exception as e: print(f'    {abslab(hs,u)} FAIL {type(e).__name__}'); continue
        rows += out
        for r in out: log('Table2abs', f'{y}/{abstag(hs,u)}/{r["kind"]}', r['IRR'], r['lo'], r['hi'], r['p'], int(PA[y].sum()))
        g = {r['kind']: r for r in out}
        f = lambda k: f'{g[k]["IRR"]:.3f}[{g[k]["lo"]:.3f},{g[k]["hi"]:.3f}]{stars(g[k]["p"]):3s}'
        print(f'    Hs≥{hs:<4g} U≥{u:<8g}{f("wind_only"):>22s}{f("wave_only"):>22s}{f("compound"):>22s}'
              f'{g["cp_vs_wo"]["IRR"]:>10.3f}{stars(g["cp_vs_wo"]["p"]):<4s}  {f("cp_vs_all"):>22s}')
T2A = pd.DataFrame(rows)
print('\n  compound = three-variable model, reference = days with neither wind nor wave at that level; cp/wo = compound relative to wind-only; cp vs all = univariate')


In [ ]:
print('='*100); print('Table 4  compound-day effect by accident type (two levels × two specifications)')
MIN_CP_EVENTS = 10                                  # not estimated if events in compound cell-months fall below this
rows = []
for hs, u in ABS_D:
    xs = abs3(hs, u); cp = xs[2]
    print(f'\n  --- {abslab(hs,u)} ---')
    for t in TYPES_ALL:
        col = f'A_{t}'; n = int(PA[col].sum())
        grp = 'weather' if t in TYPES_WX else 'control' if t in NEG_CTRL else 'other'
        if n < MIN_EVENTS: print(f'  {t:16s} n={n:5d} too few events, skipped'); continue
        n_cp = int(PA.loc[PA[cp] > 0, col].sum())
        for spec, xl in [('three', xs), ('single', [cp])]:
            m, fe = fit_ladder(col, xl, PA, [FE_ABS] + FE_REL_WI)
            if m is None: print(f'  {t:16s} [{spec}] did not converge'); continue
            irr, lo, hi, pv = row(m, cp)
            rows.append(dict(type=t, grp=grp, hs=hs, u=u, spec=spec, IRR=irr, lo=lo, hi=hi, p=pv,
                             n=n, n_cp=n_cp, fe=fe))
            log('Table4abs', f'{t}/{abstag(hs,u)}/{spec}', irr, lo, hi, pv, n)
        flag = '' if n_cp >= MIN_CP_EVENTS else f'  ⚠ only {n_cp} in cp cell-months, not estimated in figure'
        s = {r['spec']: r for r in rows if r['type'] == t and r['hs'] == hs and r['u'] == u}
        print(f'  {t:16s} n={n:5d} n_cp={n_cp:4d}  '
              + '  '.join(f'{k}={v["IRR"]:.3f}[{v["lo"]:.3f},{v["hi"]:.3f}]{stars(v["p"]):3s}' for k, v in s.items())
              + f'  {grp}{flag}')
T4B = pd.DataFrame(rows)
print('\n  three = compound vs both-low days (three-variable); single = compound vs all other days')


In [ ]:
WITH_SCAN = True          # whether a/b include the right-hand threshold-sweep panel
SHOW_ALT  = False         # whether c overlays the A_wx series
CP_SPEC   = 'three'       # compound variant for c/d: 'three' = vs both-low days; 'single' = vs all other days
CP_KIND   = {'three': 'compound', 'single': 'cp_vs_all'}[CP_SPEC]
Y_FIG = Y_MAIN
WAVE_C, WIND_C, CP_C = '#2c6e91', '#d1603d', '#7b5aa6'
INK, INK2, GREY, SPINE = '#1a1a1a', '#6b7178', '#9aa0a6', '#c9ced3'
DPI = 500
plt.rcParams['figure.dpi'] = DPI
FS = dict(tick=12, lab=13.5, row=12.5, num=11.5, letter=17, leg=12, band=11)
def is_sig(lo, hi): return not (lo <= 1 <= hi)
def letter(ax, s, x, y):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=FS['letter'], fontweight='bold', color=INK, ha='left', va='bottom')
def clean(ax, keep=('bottom',)):
    for sp in ax.spines:
        ax.spines[sp].set_visible(sp in keep); ax.spines[sp].set_color(SPINE)
    ax.set_axisbelow(True)
def blend(col, k):
    c = np.array(plt.matplotlib.colors.to_rgb(col)); return tuple(1 - k * (1 - c))

SPEC = {
 'wave': dict(col=WAVE_C, hz='wv', xlab='Incidence rate ratio',
    rows=[('Reference',  '<1.25 m',    None),         ('Moderate',   '1.25–2.5 m', 'Wv_1p25_2p5'),
          ('Rough',      '2.5–4 m',    'Wv_2p5_4'),   ('Very rough', '4–6 m',      'Wv_4_6'),
          ('High',       '6–9 m',      'Wv_6_9'),     ('Very high+', '≥9 m',       'Wv_9_p')],
    tlab=r'Threshold, $H_s$ (m)', tticks=[1.25, 2.5, 4.0, 6.0, 8.0, 9.0, 10.0],
    bands=[(1.25, 2.5, 'Mod.'), (2.5, 4, 'Rough'), (4, 6, 'V. rough'), (6, 9, 'High'), (9, 999, 'V. high')]),
 'wind': dict(col=WIND_C, hz='wd', xlab='Incidence rate ratio',
    rows=[('Reference', '<8.0 m s⁻¹',      None),           ('Bft 5', '8.0–10.8 m s⁻¹',  'Wd_8p0_10p8'),
          ('Bft 6',     '10.8–13.9 m s⁻¹', 'Wd_10p8_13p9'), ('Bft 7', '13.9–17.2 m s⁻¹', 'Wd_13p9_17p2'),
          ('Bft 8',     '17.2–20.8 m s⁻¹', 'Wd_17p2_20p8'), ('Bft ≥9', '≥20.8 m s⁻¹',    'Wd_20p8_p')],
    tlab=r'Threshold, $U_{10}$ (m s$^{-1}$)', tticks=[8.0, 13.9, 17.2, 20.8, 22.0],
    bands=[(8, 10.8, 'B5'), (10.8, 13.9, 'B6'), (13.9, 17.2, 'B7'), (17.2, 20.8, 'B8'), (20.8, 24.5, 'B9')]),
}


In [ ]:
def draw_dose(key, pl, annot):
    SP = SPEC[key]; col = SP['col']
    if WITH_SCAN:
        fig = plt.figure(figsize=(13.6, 4.9),dpi=DPI);
        axF = fig.add_axes([0.235, 0.17, 0.235, 0.70]); axS = fig.add_axes([0.66, 0.17, 0.315, 0.70])
    else:
        fig = plt.figure(figsize=(7.6, 4.9),dpi=DPI); axF = fig.add_axes([0.42, 0.17, 0.40, 0.70]); axS = None

    ax = axF; n = len(SP['rows']); tr = ax.get_yaxis_transform()
    for i, (name, rng, b) in enumerate(SP['rows']):
        yy = n - 1 - i
        if b is None:
            ax.plot(1.0, yy, 'o', ms=8, mfc='white', mec=GREY, mew=1.6, zorder=4); txt, tc = '1.000 (ref.)', INK2
        else:
            r = T1[(T1['y'] == Y_FIG) & (T1['bin'] == b)]
            if not len(r): txt, tc = '—', INK2
            else:
                r = r.iloc[0]; sig = is_sig(r['lo'], r['hi'])
                ax.errorbar(r['IRR'], yy, xerr=[[r['IRR']-r['lo']], [r['hi']-r['IRR']]], fmt='o', ms=8,
                            mfc=col if sig else 'white', mec=col, mew=1.7, ecolor=col, elinewidth=1.7, capsize=2.5, zorder=4)
                txt, tc = f"{r['IRR']:.3f} ({r['lo']:.3f}–{r['hi']:.3f})", INK
        ax.text(-0.66, yy, name, transform=tr, ha='right', va='center', fontsize=FS['row'], color=INK)
        ax.text(-0.60, yy, rng,  transform=tr, ha='left',  va='center', fontsize=FS['row'], color=INK)
        if annot: ax.text(1.08, yy, txt, transform=tr, ha='left', va='center', fontsize=FS['num'], color=tc)
    ax.axvline(1, color='#4a4a4a', lw=1.1, ls=(0, (4, 3)), zorder=1)
    ax.set_ylim(-0.6, n - 0.4); ax.set_yticks([])
    ax.set_xlim(0.97, 1.36); ax.set_xticks([1.0, 1.1, 1.2, 1.3])
    ax.set_xlabel(SP['xlab'], fontsize=FS['lab'], color=INK)
    ax.tick_params(axis='x', labelsize=FS['tick'], colors=INK2, length=3)
    clean(ax); ax.grid(axis='x', lw=.5, color='#e8ebed', zorder=0)
    if annot:
        letter(ax, pl, -0.66, 1.04)
        ax.text(1.08, 1.02, 'IRR (95% CI)', transform=ax.transAxes, fontsize=FS['num'], color=INK2, ha='left')

    if axS is not None:
        ax = axS
        s = SCAN[(SCAN['y'] == Y_FIG) & (SCAN['hazard'] == SP['hz'])].sort_values('cut')
        x0, x1 = s['cut'].min(), s['cut'].max(); pad = 0.06 * (x1 - x0); XL, XR = x0 - pad, x1 + pad
        ax.set_xlim(XL, XR)
        for k, (a0, b0, lab) in enumerate(SP['bands']):
            aa, bb = max(a0, XL), min(b0, XR)
            if bb <= aa: continue
            if k % 2 == 0: ax.axvspan(aa, bb, color='#f2f4f5', lw=0, zorder=0)
            if annot: ax.text((aa+bb)/2, .97, lab, transform=ax.get_xaxis_transform(), ha='center', va='top',
                              fontsize=FS['band'], color=GREY)
        ax.fill_between(s['cut'], (s['lo']-1)*100, (s['hi']-1)*100, color=col, alpha=.15, lw=0, zorder=2)
        ax.plot(s['cut'], (s['IRR']-1)*100, '-', lw=2.0, color=col, zorder=3)
        for _, r in s.iterrows():
            ax.plot(r['cut'], (r['IRR']-1)*100, 'o', ms=7, mfc=col if is_sig(r['lo'], r['hi']) else 'white',
                    mec=col, mew=1.6, zorder=4)
        ax.axhline(0, color=GREY, lw=1, ls=(0, (4, 3)), zorder=1)
        tt = [t for t in SP['tticks'] if XL <= t <= XR]
        ax.set_xticks(tt); ax.set_xticklabels([f'{t:g}' for t in tt])
        ax.set_xlabel(SP['tlab'], fontsize=FS['lab'], color=INK)
        ax.set_ylabel('Change in accident rate (%)', fontsize=FS['lab'], color=INK)
        ax.tick_params(labelsize=FS['tick'], colors=INK2, length=3)
        clean(ax, keep=('bottom', 'left')); ax.grid(axis='y', lw=.5, color='#e8ebed', zorder=0)
    if annot:
        fig.legend(handles=[Line2D([], [], marker='o', ls='', ms=8, mfc='#4a4a4a', mec='#4a4a4a', label='95% CI excludes null'),
                            Line2D([], [], marker='o', ls='', ms=8, mfc='white', mec='#4a4a4a', mew=1.6, label='95% CI includes null')],
                   loc='lower center', bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False, fontsize=FS['leg'])
    plt.show()


In [ ]:
LAD_ROW = {(1.25, 8.0): ('Moderate · Bft 5',   'Hs ≥ 1.25 m, U₁₀ ≥ 8.0 m s⁻¹'),
           (2.5, 10.8): ('Rough · Bft 6',      'Hs ≥ 2.5 m, U₁₀ ≥ 10.8 m s⁻¹'),
           (4.0, 13.9): ('Very rough · Bft 7', 'Hs ≥ 4 m, U₁₀ ≥ 13.9 m s⁻¹'),
           (6.0, 17.2): ('High · Bft 8',       'Hs ≥ 6 m, U₁₀ ≥ 17.2 m s⁻¹'),
           (9.0, 20.8): ('Very high · Bft 9',  'Hs ≥ 9 m, U₁₀ ≥ 20.8 m s⁻¹')}

def draw_compound(pl, annot):
    series = [(Y_MAIN, 'All accidents', CP_C, 'o', 0.0)]
    if SHOW_ALT: series = [(Y_MAIN, 'All accidents', CP_C, 'o', +0.17), (Y_ALT, 'Weather-related', '#b59ad6', 'D', -0.17)]
    n = len(ABS_LADDER)
    fig = plt.figure(figsize=(12.4, 0.95 * n + 1.9),dpi=DPI)
    ax = fig.add_axes([0.40, 0.20, 0.33, 0.70]); tr = ax.get_yaxis_transform()
    xmax = 0
    for i, thr in enumerate(ABS_LADDER):
        yy = n - 1 - i
        if i % 2 == 0: ax.axhspan(yy - .5, yy + .5, color='#f4f5f6', lw=0, zorder=0)
        name, rng = LAD_ROW[thr]
        ax.text(-0.72, yy, name, transform=tr, ha='right', va='center', fontsize=FS['row'], color=INK)
        ax.text(-0.66, yy, rng,  transform=tr, ha='left',  va='center', fontsize=FS['row'] - 1, color=INK2)
        for k, (y, lab, col, mk, off) in enumerate(series):
            r = T2A[(T2A.y == y) & (T2A.hs == thr[0]) & (T2A.u == thr[1]) & (T2A.kind == CP_KIND)]
            if not len(r): continue
            r = r.iloc[0]; c, l, h = (r.IRR-1)*100, (r.lo-1)*100, (r.hi-1)*100; xmax = max(xmax, h)
            sig = is_sig(r.lo, r.hi)
            ax.errorbar(c, yy + off, xerr=[[c-l], [h-c]], fmt=mk, ms=6, mfc=col if sig else 'white',
                        mec=col, mew=1.9, ecolor=col, elinewidth=1.8, capsize=4.5, capthick=1.6, zorder=4)
            if annot and k == 0:
                days = int(PA[abs3(*thr)[2]].sum())
                ax.text(1.06, yy + 0.16, f'{r.IRR:.3f} ({r.lo:.3f}–{r.hi:.3f})', transform=tr,
                        ha='left', va='center', fontsize=FS['num'], color=INK)
                ax.text(1.06, yy - 0.20, f'{days:,} compound days', transform=tr,
                        ha='left', va='center', fontsize=FS['num'] - 1.5, color=GREY)
    ax.axvline(0, color='#4a4a4a', lw=1.2, ls=(0, (4, 3)), zorder=1)
    ax.set_ylim(-.6, n - .4); ax.set_yticks([])
    ax.set_xlim(-8, np.ceil(xmax / 10) * 10 + 4)
    ax.set_xlabel('Change in accident rate per compound-extreme day (%)', fontsize=FS['lab'], color=INK)
    ax.tick_params(axis='x', labelsize=FS['tick'], colors=INK2, length=3); ax.tick_params(axis='y', length=0)
    clean(ax); ax.grid(axis='x', lw=.5, color='#e8ebed', zorder=0)
    if annot:
        letter(ax, pl, -0.72, 1.04)
        ax.text(1.06, 1.02, 'IRR (95% CI)', transform=ax.transAxes, fontsize=FS['num'], color=INK2, ha='left')
        hs_ = [Line2D([], [], marker=mk, ls='', ms=10, mfc=col, mec=col, label=lab) for _, lab, col, mk, _ in series]
        hs_.append(Line2D([], [], marker='o', ls='', ms=10, mfc='white', mec='#4a4a4a', mew=1.8, label='95% CI includes 0'))
        fig.legend(handles=hs_, loc='lower center', bbox_to_anchor=(0.5, 0.0), ncol=len(hs_), frameon=False, fontsize=FS['leg'])
    plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
TILE_CMAP = LinearSegmentedColormap.from_list('purp', ['#f8f4fb', '#c9a8dd', '#8f4fb3', '#5b2d86'])
SIG_EDGE  = '#d62728'
NAME = {'Fire_Explosion': 'Fire / explosion', 'Hull_Damage': 'Hull damage', 'Piracy_Other': 'Piracy / other',
        'Machinery': 'Machinery', 'Grounding': 'Grounding', 'Collision': 'Collision', 'Contact': 'Contact',
        'Foundered': 'Foundered', 'Misc': 'Miscellaneous'}
# row order; None = weather types by severe-level IRR descending, then other, then negative controls
D_TYPES = ['Foundered', 'Grounding', 'Collision', 'Hull_Damage', 'Fire_Explosion', 'Machinery', 'Piracy_Other']
def pnum(v):
    """% number format: show 0.0 if it rounds to 0 (avoid -0.0), use − for minus"""
    v = round(v, 1); return '0.0' if v == 0 else f'{v:+.1f}'.replace('-', '−')

def draw_types(pl, annot):
    d = T4B[T4B.spec == CP_SPEC]
    sev = d[(d.hs == ABS_D[1][0]) & (d.u == ABS_D[1][1])].set_index('type')
    if D_TYPES is None:
        order = (sev[sev.grp == 'weather'].sort_values('IRR', ascending=False).index.tolist()
                 + sev[sev.grp == 'other'].index.tolist() + sev[sev.grp == 'control'].index.tolist())
    else:
        order = [t for t in D_TYPES if t in sev.index]
    cols = ABS_D; nr, nc = len(order), len(cols)
    cells = {}
    for t in order:
        for thr in cols:
            r = d[(d.type == t) & (d.hs == thr[0]) & (d.u == thr[1])]
            if len(r): cells[(t, thr)] = r.iloc[0]
    vmax = max((v.IRR - 1) * 100 for v in cells.values() if v.n_cp >= MIN_CP_EVENTS)

    fig, ax = plt.subplots(figsize=(3.3 * nc + 3.6, 1.15 * nr + 1.6), dpi=DPI)
    fig.subplots_adjust(left=0.36, right=0.98, top=0.85, bottom=0.05)
    for i, t in enumerate(order):
        yy = nr - 1 - i
        for j, thr in enumerate(cols):
            r = cells.get((t, thr))
            if r is None: continue
            if r.n_cp < MIN_CP_EVENTS:
                ax.add_patch(mpatches.Rectangle((j - .46, yy - .42), .92, .84, facecolor='#f2f3f4', edgecolor='none', zorder=2))
                ax.text(j, yy, f'n < {MIN_CP_EVENTS}', ha='center', va='center', fontsize=FS['row'], color=INK2, zorder=3)
                continue
            c = (r.IRR - 1) * 100; sig = is_sig(r.lo, r.hi)
            face = TILE_CMAP(np.clip(c, 0, vmax) / vmax)
            ax.add_patch(mpatches.Rectangle((j - .46, yy - .42), .92, .84, facecolor=face,
                                            edgecolor=SIG_EDGE if sig else 'none', lw=2.6 if sig else 0, zorder=2))
            lum = 0.299 * face[0] + 0.587 * face[1] + 0.114 * face[2]
            tc = 'white' if lum < 0.62 else INK
            ax.text(j, yy + .13, f'{pnum(c)}%', ha='center', va='center', fontsize=FS['row'] + 2,
                    fontweight='bold' if sig else 'normal', color=tc, zorder=3)
            ax.text(j, yy - .20, f'[{pnum((r.lo - 1) * 100)}, {pnum((r.hi - 1) * 100)}]', ha='center', va='center',
                    fontsize=FS['row'] - 1, color=tc, zorder=3)
        lab = NAME.get(t, t) + (f'   ({int(sev.loc[t, "n"]):,})' if annot else '')
        ax.text(-0.62, yy, lab, ha='right', va='center', fontsize=FS['row'] + 2, color=INK)
    for j, thr in enumerate(cols):
        ax.text(j, nr - .30, LAD_NAME[thr], ha='center', va='bottom', fontsize=FS['row'] + 1, color=INK, linespacing=1.25)
    ax.set_xlim(-.6, nc - .4); ax.set_ylim(-.6, nr + .6); ax.axis('off')
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    sm = ScalarMappable(cmap=TILE_CMAP, norm=Normalize(vmin=0, vmax=vmax)); sm.set_array([])
    cax = fig.add_axes([0.40, 0.0, 0.40, 0.025])                      # [left, bottom, width, height], figure coords
    cb = fig.colorbar(sm, cax=cax, orientation='horizontal', extend='min')   # left triangle: negatives share the colour of 0
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=FS['tick'] - 1, colors=INK2, length=3)
    step = 25 if vmax <= 100 else 50; ticks = np.arange(0, vmax + 1e-9, step)   # ticks follow vmax, stay within the colorbar
    cb.set_ticks(ticks); cb.set_ticklabels(['≤0'] + [f'+{v:.0f}%' for v in ticks[1:]])
    if annot: cb.set_label('Change in accident rate per compound-extreme day (%)', fontsize=FS['num'], color=INK)

    if annot:
        letter(ax, pl, -0.55, 0.99)
        ax.text(nc - .4, -0.62, f'Values: % change [95% CI];  red outline: CI excludes 0;  grey: fewer than {MIN_CP_EVENTS} accidents on compound days',
                ha='right', va='top', fontsize=FS['num'], color=INK2)
    plt.show()


In [ ]:
for ANNOT in (True, False):
    print(f'\n{"="*30}  {"with text" if ANNOT else "no text"}  {"="*30}')
    draw_dose('wave', 'a', ANNOT)
    draw_dose('wind', 'b', ANNOT)
    draw_compound('c', ANNOT)
    draw_types('d', ANNOT)


In [ ]:
for ANNOT in (True, False):
    print(f'\n{"="*30}  {"with text" if ANNOT else "no text"}  {"="*30}')
    draw_dose('wave', 'a',False)
    draw_dose('wind', 'b',False)
    draw_compound('c', False)
    draw_types('d', False)
